# DATALUS Generative Studio: Simulação de Cenários e GenAI Aplicada

Este notebook demonstra o **DATALUS Generative Studio**, com foco em tarefas de IA Generativa aplicada usando um modelo de difusão pré-treinado.
Exploramos como Cientistas de Dados e Gestores de Políticas Públicas podem aproveitar dados sintéticos para simulação de cenários,
balanceamento de classes minoritárias, inpainting tabular e aumento de dados avançado.

In [ ]:
# ============================================================
# CONFIGURAÇÃO DO DATASET
# ============================================================
# Defina USE_REAL_DATA como True para baixar registros reais
# do SIH-SUS dos servidores FTP do DATASUS. Defina como False
# para gerar um dataset sintético totalmente configurável.
USE_REAL_DATA = False

import polars as pl
import numpy as np

SYNTHETIC_CONFIG = {
    'n_rows': 100_000,
    'seed': 42,
    'columns': {
        'MORTE': {
            'dtype': pl.Int64,
            'generator': lambda n, rng: rng.choice([0, 1], n, p=[0.92, 0.08]).astype(np.int64),
        },
        'IDADE': {
            'dtype': pl.Float64,
            'generator': lambda n, rng: rng.integers(0, 100, n).astype(np.float64),
        },
        'SEXO': {
            'dtype': pl.Utf8,
            'generator': lambda n, rng: rng.choice(['0', '1'], n),
        },
        'DIAS_PERMANENCIA': {
            'dtype': pl.Float64,
            'generator': lambda n, rng: np.clip(rng.poisson(5, n), 0, 100).astype(np.float64),
        },
        'VALOR_TOTAL': {
            'dtype': pl.Float64,
            'generator': lambda n, rng: np.round(np.exp(rng.normal(7, 1.5, n)), 2),
        },
        'PROCEDIMENTO_PRINCIPAL': {
            'dtype': pl.Utf8,
            'generator': lambda n, rng: rng.choice(['AB', 'CD', 'EF', 'GH', 'IJ'], n),
        },
        'MUNICIPIO_MOV': {
            'dtype': pl.Utf8,
            'generator': lambda n, rng: rng.choice(['3550308', '3304557', '3106200'], n),
        },
    },
    'target_column': 'MORTE',
}

## 1. Configuração do Ambiente e Verificação de Artefatos
Detectando o ambiente e verificando os pré-requisitos. Se os artefatos de treinamento estiverem ausentes, executamos um passe de treinamento rapido.

**Google Colab:** Após a conclusão da célula de instalação abaixo, o Google Colab pode reiniciar automaticamente o runtime devido a mudanças nas dependências. Se isso ocorrer, ou se você vir a mensagem de instalação concluída, reinicie o runtime manualmente via `Ambiente de execução > Reiniciar e executar tudo` e reexecute todas as células a partir da primeira célula. O arquivo `.installed` impedirá a reinstalação na próxima execução.

**WORKING_DIR:** Se você encontrar um erro "No such file or directory" em qualquer comando `datalus`, a variável `WORKING_DIR` pode não estar sendo resolvida corretamente. Nesse caso, defina `WORKING_DIR` manualmente para o caminho absoluto do seu diretório de trabalho na célula de configuração do ambiente abaixo.

In [ ]:
import os
import sys
from pathlib import Path

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_studio'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_studio'
    return './datalus_studio'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Diretório de trabalho: {WORKING_DIR}')

# Verificar se WORKING_DIR pode ser escrito
try:
    test_file = Path(WORKING_DIR) / '.write_test'
    test_file.touch()
    test_file.unlink()
except (OSError, PermissionError):
    print(f'AVISO: Não é possível escrever em {WORKING_DIR}')
    print('Defina WORKING_DIR manualmente:')
    print('  WORKING_DIR = "/path/to/your/directory"')

INSTALLED = Path(WORKING_DIR) / '.installed'

if not INSTALLED.exists():
    if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        !git clone --depth 1 https://github.com/emanuellcs/datalus.git "{WORKING_DIR}/source" 2>/dev/null || true
        %cd "{WORKING_DIR}/source"
        !pip install --quiet -e '.[dev]' pysus polars matplotlib seaborn nest_asyncio 2>&1 | tail -5
        sys.path.insert(0, f'{WORKING_DIR}/source')
        INSTALLED.touch()
        print()
        print('=' * 60)
        print('  INSTALAÇÃO CONCLUÍDA')
        print('  Reinicie o runtime manualmente:')
        print('    Ambiente de execução > Reiniciar e executar tudo')
        print('  Em seguida, reexecute todas as células desde o início.')
        print('=' * 60)
    else:
        INSTALLED.touch()
        print('Ambiente local. Dependências consideradas instaladas.')
else:
    src = Path(WORKING_DIR) / 'source'
    if src.is_dir():
        sys.path.insert(0, str(src))
    print('Dependências já instaladas.')

checkpoint_path = Path(f'{WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt')
if not checkpoint_path.exists():
    print('Artefatos de treinamento não encontrados. Executando passagem de treinamento rápido...')
    from pysus import sih, sim, sinan
    from datetime import datetime
    import urllib.request
    import asyncio

    STATES = ['SP', 'RJ', 'MG', 'PR', 'RS', 'BA', 'DF']

    def fmt_tier(label, state, year, month, count):
    return f'Camada {label}: {state}/{year}/{month} ({count} registros)'

    def try_pysus():
        now = datetime.now()
        sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
        for state in STATES:
            for y in range(sy, sy - 2, -1):
                for m in range(sm if y == sy else 12, 0, -1):
                    try:
                        df = sih(state=state, year=y, month=[m], group='RD', as_dataframe=True)
                        if df is not None and len(df) > 0 and 'MORTE' in df.columns:
                            print(fmt_tier('1: SIH-RD', state, y, m, len(df)))
                            return pl.from_pandas(df)
                    except Exception:
                        continue
            for y in range(sy, sy - 2, -1):
                try:
                    df = sim(state=state, year=y, as_dataframe=True)
                    if df is not None and len(df) > 0:
                        print(fmt_tier('2: SIM', state, y, 'all', len(df)))
                        df['MORTE'] = 1
                        return pl.from_pandas(df)
                except Exception:
                    continue
        for y in range(sy, sy - 2, -1):
            try:
                df = sinan(disease='deng', year=y, as_dataframe=True)
                if df is not None and len(df) > 0:
                    print(fmt_tier('3: SINAN-deng', 'BR', y, 'all', len(df)))
                    if 'MORTE' not in df.columns:
                        df['MORTE'] = 0
                    return pl.from_pandas(df)
            except Exception:
                continue
        return None

    def try_direct_ftp():
        try:
            import nest_asyncio
            nest_asyncio.apply()
        except ImportError:
            return None
        from pysus.api.extensions import DBC
        now = datetime.now()
        sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
        base = 'ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/199301_/Dados/'
        for state in ['SP', 'RJ', 'MG']:
            for y in range(sy, sy - 2, -1):
                for m in range(sm if y == sy else 12, 0, -1):
                    fname = f'RD{state}{y % 100:02d}{m:02d}.dbc'
                    try:
                        with urllib.request.urlopen(base + fname, timeout=30) as r:
                            dbc_path = Path('/tmp') / fname
                            dbc_path.write_bytes(r.read())
                        df = asyncio.run(DBC(path=dbc_path).load())
                        if df is not None and len(df) > 0:
                            print(fmt_tier('4: FTP direto', state, y, m, len(df)))
                            return pl.from_pandas(df)
                    except Exception:
                        continue
        return None

    if USE_REAL_DATA:
        print('Baixando dados clínicos do DATASUS...')
        df = try_pysus()
        if df is None:
            df = try_direct_ftp()
        if df is None:
            raise RuntimeError(
                'Não foi possível baixar dados reais do DATASUS. '
                'Defina USE_REAL_DATA = False na célula de configuração '
                'para usar um dataset sintético.'
            )
    else:
        print(f'Gerando dataset sintético ({SYNTHETIC_CONFIG["n_rows"]} linhas)...')
        rng = np.random.default_rng(SYNTHETIC_CONFIG['seed'])
        n = SYNTHETIC_CONFIG['n_rows']
        data = {}
        for name, cfg in SYNTHETIC_CONFIG['columns'].items():
            data[name] = cfg['generator'](n, rng)
        df = pl.DataFrame(
            data,
            schema={name: cfg['dtype'] for name, cfg in SYNTHETIC_CONFIG['columns'].items()},
        )

    df = df.with_columns(
        pl.col('MORTE').cast(pl.Int64).fill_null(0)
    )

    cols_to_keep = [
        c for c in df.columns
        if not c.startswith(('N_AIH', 'IDENT', 'CNS', 'CPF', 'CNPJ'))
    ]
    df = df.select(
        ['MORTE'] + [c for c in cols_to_keep if c != 'MORTE']
    ).head(5000)
    df.write_parquet(f'{WORKING_DIR}/raw_sample.parquet')
    
    !datalus ingest {WORKING_DIR}/raw_sample.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE
    !datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 1 --batch-size 256 --max-steps 1000 --gpu 0
else:
    print('Verificado: Artefatos de treinamento prontos.')


## 2. Geração Ab-Initio e Análise de Distribuição (KDE)
Gerando um grande conjunto de dados sintéticos e comparando sua distribuição estatística com os dados reais usando estimativa de densidade de kernel (KDE).

In [ ]:
!datalus sample {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet --n-records 10000 --ddim-steps 50 --seed 42 --cfg-scale 1.0

import matplotlib.pyplot as plt
import seaborn as sns

real = pl.read_parquet(f'{WORKING_DIR}/processed.parquet')
synth = pl.read_parquet(f'{WORKING_DIR}/synthetic.parquet')

plt.figure(figsize=(12, 6))
sns.kdeplot(data=real['IDADE'], label='Real Data', fill=True, alpha=0.3)
sns.kdeplot(data=synth['IDADE'], label='Synthetic Data', fill=True, alpha=0.3)
plt.title('Kernel Density Estimation: Real vs Synthetic (Age / IDADE)')
plt.xlabel('Age (Years)')
plt.ylabel('Density')
plt.legend()
plt.show()

## 3. Aumento de Dados Avançado (Augmentation)
Demonstrando como o aumento sintético evita o overfitting em tarefas subsequentes de machine learning.

In [ ]:
seed = synth.sample(n=1000)
seed.write_parquet(f'{WORKING_DIR}/seed_augment.parquet')

!datalus augment {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/seed_augment.parquet {WORKING_DIR}/augmented.parquet --n-records 1000 --ddim-steps 50 --seed 42 --cfg-scale 1.0

## 4. Justiça Algorítmica: Balanceamento de Classes Minoritárias
Lidando com o desequilíbrio de classes (ex: doenças raras ou demografias específicas) através de sobreamostragem sem duplicação de linhas.

In [ ]:
!datalus balance {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/balanced.parquet MORTE '{"1": 5000}' --ddim-steps 50 --seed 42 --cfg-scale 1.0 --max-attempts 10

balanced = pl.read_parquet(f'{WORKING_DIR}/balanced.parquet')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
real['MORTE'].value_counts().to_pandas().plot(kind='bar', x='MORTE', y='count', ax=ax1, color='salmon')
ax1.set_title('Original Class Imbalance (Real)')
balanced['MORTE'].value_counts().to_pandas().plot(kind='bar', x='MORTE', y='count', ax=ax2, color='skyblue')
ax2.set_title('Balanced Class Distribution (Synthetic)')
plt.show()

## 5. Inpainting Tabular (Imputação de Dados Faltantes)
Usando inferência no estilo RePaint para preencher condicionalmente valores nulos enquanto preserva os campos observados.

In [ ]:
mask = np.random.rand(100) > 0.5
incomplete = real.head(100).with_columns(
    pl.when(pl.lit(mask)).then(None).otherwise(pl.col('IDADE')).alias('IDADE')
)
incomplete.write_parquet(f'{WORKING_DIR}/incomplete.parquet')

!datalus inpaint {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/incomplete.parquet {WORKING_DIR}/inpainted.parquet --ddim-steps 50 --jump-length 10 --jump-n-sample 10 --seed 42

## 6. Intervenções Contrafactuais e Simulação de Políticas
Simulando cenários "E se" ao modificar caracteristicas do paciente e permitindo que o modelo regenere o registro clínico de forma coerente.

In [ ]:
!datalus counterfactual {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/counterfactual.parquet '{"SEXO": "1"}' --ddim-steps 50 --seed 42